In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# シンプルなフォント設定
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Helvetica']

class StretchModelAnalyzer:
    def __init__(self, neck_shape_mode=2):
        self.scale_length = 628.0
        self.string_spacing = 10.8
        self.f_max = 22
        
        # ポジション最大値 (基準)
        self.p_max = 16
        
        # 指パラメータ
        self.finger_lengths = {1: 70.0, 2: 75.0, 3: 70.0, 4: 50.0}
        self.knuckle_dist = {1: 0.0, 2: 20.0, 3: 40.0, 4: 55.0}
        
        # 難易度係数
        self.finger_difficulty = {1: 1.0, 2: 1.0, 3: 1.1, 4: 1.5}
        
        # 距離のハードリミット倍率
        self.limit_ratios = {1: 1.3, 2: 1.05, 3: 1.05, 4: 1.1}
        
        # 基本有効長係数
        self.base_factors = {1: 0.7, 2: 0.90, 3: 0.85, 4: 0.95}

        # --- ネック形状と減少率の設定 ---
        # 16F(p_max)時点での最大減少率 (1:15%, 2:30%, 3:45%)
        self.neck_reduction_rates = {1: 0.15, 2: 0.225, 3: 0.30}
        self.current_max_reduction = self.neck_reduction_rates.get(neck_shape_mode, 0.30)
        
        # 「ポジション12で約0.05(5%)になる」ための指数kを計算
        # 式: 0.05 = Max * (12/16)^k
        target_pos_ratio = 12.0 / 16.0
        target_reduction = 0.03
        
        if self.current_max_reduction > target_reduction:
            self.curve_exponent = math.log(target_reduction / self.current_max_reduction) / math.log(target_pos_ratio)
        else:
            self.curve_exponent = 1.0 

        # 指ごとの減少率倍率
        self.finger_multipliers = {1: 0.0, 2: 0.0, 3: 1.2, 4: 1.3}

        # フレット位置計算 (ナットからの距離)
        self.fret_pos = [
            self.scale_length - (self.scale_length / (2 ** (f / 12.0)))
            for f in range(self.f_max + 1)
        ]

    def get_dist_from_nut(self, fret):
        """指定フレットのナットからの距離を返す"""
        if fret > self.f_max: fret = self.f_max
        if fret < 0: fret = 0
        return self.scale_length - self.fret_pos[fret]

    def calculate_neck_factor(self, position, finger_idx):
        """
        指定ポジション・指における有効指長の係数(Neck Factor)を計算
        ポジション(0->16)に対して指数関数的に減少率を適用
        """
        # 進行度 (Pos 0 -> Pos 16 = 1.0)
        pos_val = float(position)
        if pos_val < 0: pos_val = 0.0
        
        ratio = pos_val / self.p_max
        
        # 16F以降は最大減少率を維持
        if ratio > 1.0:
            ratio = 1.0
            
        # 減少率の計算: Max * (ratio ^ k) * 指倍率
        base_reduction = self.current_max_reduction * (ratio ** self.curve_exponent)
        
        # 指ごとの倍率を適用
        reduction = base_reduction * self.finger_multipliers.get(finger_idx, 1.0)
        
        # 係数算出 (1.0 - 減少分), 安全のため0.1でクリップ
        factor = max(0.1, 1.0 - reduction)
        
        return factor

    def calculate_min_stretch_cost_detailed(self, hand_pos, finger_idx, string, fret):
        if fret > self.f_max: return None, None, None
        if fret < hand_pos: return None, None, None

        pivot_x = self.fret_pos[hand_pos]
        pivot_y = 0.0 

        target_x = self.fret_pos[fret]
        target_y = (string - 1) * self.string_spacing

        min_cost = float('inf')
        best_angle = None
        best_finger_angle = None
        valid_solution_found = False

        # 手首角度探索 (-90度 〜 0度)
        for angle_deg in range(-90, 1, 2):
            theta_rad = math.radians(angle_deg)

            # 1. ナックル位置計算
            d = self.knuckle_dist[finger_idx]
            k_x = pivot_x + (d * math.cos(theta_rad))
            k_y = pivot_y + (d * math.sin(theta_rad))

            # 2. 距離計算
            v_x = target_x - k_x
            v_y = target_y - k_y
            dist = math.sqrt(v_x**2 + v_y**2)

            # 3. 有効指長の計算 (ポジション基準に変更: hand_posを使用)
            neck_factor = self.calculate_neck_factor(hand_pos, finger_idx)
            eff_len = self.finger_lengths[finger_idx] * self.base_factors[finger_idx] * neck_factor

            # ハードリミットチェック
            if eff_len > 0 and dist > eff_len * self.limit_ratios[finger_idx]:
                continue 

            # 4. 指の角度計算
            n_x = -math.sin(theta_rad)
            n_y = math.cos(theta_rad)
            
            dot_prod = v_x * n_x + v_y * n_y
            cos_delta = dot_prod / dist if dist > 0.001 else 1.0
            cos_delta = max(-1.0, min(1.0, cos_delta))
            delta_abs_deg = math.degrees(math.acos(cos_delta))

            cross_prod = n_x * v_y - n_y * v_x
            is_low_fret_side = (cross_prod > 0)
            delta_signed_deg = delta_abs_deg if is_low_fret_side else -delta_abs_deg

            # 角度制限
            if is_low_fret_side:
                limit_angle = 15.0
                relax_add = 5.0
            else:
                limit_angle = 30.0
                relax_add = 10.0 
            
            if finger_idx == 4: # 小指緩和
                limit_angle += 5.0 
            
            is_close_distance = False
            ratio = 0.0
            if eff_len > 0:
                ratio = dist / eff_len
                if ratio < 0.5: 
                    limit_angle += relax_add
                    is_close_distance = True
            
            if delta_abs_deg > limit_angle:
                continue 

            # 5. コスト計算 (距離)
            dist_cost = 0.0
            dist_slope = 20.0 
            
            if ratio < 0.3:
                dist_cost = (0.3 - ratio) * 5.0
            elif ratio <= 0.6:
                dist_cost = 0.0
            else:
                dist_cost = (ratio - 0.6) * dist_slope
                if ratio > 1.0:
                    dist_cost += (ratio - 1.0) * 30.0

            # 6. コスト計算 (角度)
            comfort_angle = 5.0
            coeff_negative = 1.0
            coeff_positive = 1.5
            angle_cost = 0.0

            if delta_signed_deg > comfort_angle:
                angle_cost = (delta_signed_deg - comfort_angle) * coeff_positive
            elif delta_signed_deg < -comfort_angle:
                angle_cost = (abs(delta_signed_deg) - comfort_angle) * coeff_negative
            
            if is_close_distance:
                angle_cost *= 0.5

            total_current_cost = (dist_cost + angle_cost)

            if total_current_cost < min_cost:
                min_cost = total_current_cost
                best_angle = angle_deg
                best_finger_angle = delta_signed_deg
                valid_solution_found = True

        if valid_solution_found:
            return min_cost, abs(best_angle), best_finger_angle
        else:
            return None, None, None

def plot_heatmap_v17_fixed(analyzer, hand_pos=12):
    fingers = {1: "Index", 2: "Middle", 3: "Ring", 4: "Pinky"}
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    axes = axes.flatten()
    
    max_plot_fret = min(hand_pos + 8, analyzer.f_max + 1)
    fret_range = range(hand_pos, max_plot_fret)
    string_range = range(1, 7)

    for i, (f_idx, f_name) in enumerate(fingers.items()):
        cost_data = []
        for s in string_range:
            cost_row = []
            for f in fret_range:
                cost, _, _ = analyzer.calculate_min_stretch_cost_detailed(hand_pos, f_idx, s, f)
                cost_row.append(cost)
            cost_data.append(cost_row)
        
        df_cost = pd.DataFrame(cost_data, index=list(string_range), columns=list(fret_range)).astype(float)
        
        sns.heatmap(df_cost, ax=axes[i], annot=True, fmt=".1f", cmap="RdYlGn_r", 
                    vmin=0, vmax=30, cbar=True, square=True,
                    mask=df_cost.isnull(), linewidths=1, linecolor='lightgray', 
                    annot_kws={"size": 14, "family": "sans-serif"})
        
        axes[i].set_title(f"{f_name} (Pos {hand_pos})", fontsize=13)
        axes[i].set_ylabel("String", fontsize=11)
        axes[i].set_xlabel("Fret", fontsize=11)
        axes[i].tick_params(labelsize=10)
    
    plt.suptitle(f"Stretch Cost Heatmap (Mode={analyzer.current_max_reduction*100:.0f}% Max@Pos16, Pos12~5% Exp={analyzer.curve_exponent:.2f})", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# --- 実行ブロック ---
if __name__ == "__main__":
    # モード設定: 1=15%, 2=30%, 3=45% (いずれもPos 12で約5%になるようカーブ調整)
    mode = 3 
    analyzer = StretchModelAnalyzer(neck_shape_mode=mode)
    
    # 任意のポジションで表示 (例: 12ポジション)
    plot_heatmap_v17_fixed(analyzer, hand_pos=12)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def draw_geometric_model_final_inner_theta():
    # --- フォントとスタイルの設定 ---
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['mathtext.fontset'] = 'cm' 
    
    # 文字サイズ設定
    FS_TICK = 22        # 弦番号・フレット番号
    FS_MATH = 34        # D, Theta

    # --- パラメータ設定 ---
    scale_length = 648.0
    string_spacing = 10.0
    
    pos_index = 5
    target_fret = 7
    target_string = 4 # 4弦
    
    active_finger_h = 3 
    finger_base_distances = {1: 0, 2: 20, 3: 40, 4: 60} 
    
    wrist_angle_deg = -25
    theta = np.deg2rad(wrist_angle_deg)
    
    # 線の太さ
    LW_STRING = 1.5
    LW_FRET = 1.0
    LW_MODEL = 2.5
    
    # 色設定
    COLOR_TEXT = '#000000'
    COLOR_DIM = '#444444' 
    COLOR_MODEL_LINE = '#000000'
    COLOR_PIVOT = '#D00000' 
    COLOR_TARGET = '#0000C0' 

    # --- 座標計算関数 ---
    def get_fret_x(fret_num):
        return scale_length * (1 - (1/2)**(fret_num / 12.0))

    def get_string_y(string_num):
        return (string_num - 1) * string_spacing

    x_pivot = get_fret_x(pos_index)
    y_pivot = 0 
    x_target = get_fret_x(target_fret)
    y_target = get_string_y(target_string)

    knuckles = {}
    for h, d_h in finger_base_distances.items():
        k_x = x_pivot + d_h * np.cos(theta)
        k_y = y_pivot + d_h * np.sin(theta)
        knuckles[h] = (k_x, k_y)

    x_knuckle_active, y_knuckle_active = knuckles[active_finger_h]

    # --- 描画セットアップ ---
    fig, ax = plt.subplots(figsize=(12, 9))
    
    # 表示範囲 (4F - 8F)
    fret_view_start, fret_view_end = 4, 8
    x_min = get_fret_x(fret_view_start) - 10
    x_max = get_fret_x(fret_view_end) + 10
    y_min_limit = -45 
    y_max_limit = 65  

    # --- 背景：弦 ---
    for s in range(1, 7):
        y = get_string_y(s)
        ax.plot([x_min, x_max], [y, y], color='#888888', linewidth=LW_STRING, zorder=1)
        # 弦番号
        ax.text(x_min - 1, y, f'{s} str', va='center', ha='right', 
                fontsize=FS_TICK, color=COLOR_DIM, family='serif')

    # --- 背景：フレット ---
    y_string_1 = get_string_y(1)
    y_string_6 = get_string_y(6)
    
    for f in range(fret_view_start, fret_view_end + 1):
        x = get_fret_x(f)
        ax.plot([x, x], [y_string_1, y_string_6], color='#666666', linewidth=LW_FRET, zorder=1)
        # フレット番号
        ax.text(x, y_string_1 - 8, f'{f} F', ha='center', fontsize=FS_TICK, fontweight='normal', color=COLOR_TEXT)

    # --- 幾何要素 ---

    # 1. ハンドベースライン
    k_1 = knuckles[1]
    k_4 = knuckles[4]
    ax.plot([k_1[0], k_4[0]], [k_1[1], k_4[1]], 
            color=COLOR_MODEL_LINE, linestyle='--', linewidth=LW_MODEL)

    # 2. ナックル (Points only)
    ax.plot(x_pivot, y_pivot, 'o', color=COLOR_PIVOT, markersize=16, zorder=15)
    for h in range(2, 5):
        kx, ky = knuckles[h]
        ax.plot(kx, ky, 'o', color='black', markerfacecolor='white', 
                markersize=12, markeredgewidth=2.0, zorder=15)

    # 3. ターゲット (Point only)
    ax.plot(x_target, y_target, 'x', color=COLOR_TARGET, markersize=18, markeredgewidth=4.0, zorder=15)

    # 4. 距離 D (Arrow + Label)
    ax.annotate('', xy=(x_target, y_target), xytext=(x_knuckle_active, y_knuckle_active),
                arrowprops=dict(arrowstyle='->', color='black', linewidth=LW_MODEL))
    
    mid_x = (x_target + x_knuckle_active) / 2
    mid_y = (y_target + y_knuckle_active) / 2
    ax.text(mid_x, mid_y, r'$D$', fontsize=FS_MATH, ha='center', va='center',
            bbox=dict(facecolor='white', edgecolor='none', pad=4.0))

    # 5. 角度 theta (内側に調整)
    # 基準線 (x軸平行)
    ax.plot([x_pivot, x_pivot + 30], [y_pivot, y_pivot], 'k:', linewidth=1.5)
    
    # 円弧 (半径を小さく: 25 -> 15)
    arc_radius = 15
    arc = patches.Arc((x_pivot, y_pivot), arc_radius*2, arc_radius*2, 
                      angle=0, theta1=wrist_angle_deg, theta2=0, color='black', linewidth=2.0)
    ax.add_patch(arc)
    


    # --- グラフ整形 ---
    ax.set_aspect('equal')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min_limit, y_max_limit)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    draw_geometric_model_final_inner_theta()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

# =============================================================================
# 1. 物理モデル計算ロジック (Grimes 2014 Eq.7 Implementation)
# =============================================================================
class PhysicsCostAnalyzer:
    def __init__(self):
        # ギターパラメータ
        self.scaleLength = 635.0
        self.f_max = 22  # 22フレットまで
        self.s_max = 6
        
        # --- 物理パラメータ (Grimes Model / OPL準拠) ---
        # ヤング率 E [GPa] (1-3弦:Plain Steel, 4-6弦:Wound)
        self.E_GPa = {
            1: 179.0, 2: 188.0, 3: 177.0, 
            4: 62.0,  5: 42.0,  6: 33.0
        }
        
        # 断面積 Area [m^2] (Gauge .010 - .046)
        self.Area_m2 = {
            1: 5.13e-8, 2: 8.56e-8, 3: 1.48e-7,
            4: 3.55e-7, 5: 6.79e-7, 6: 1.10e-6
        }
        
        # 張力 Tension [N] (約 60N - 80N)
        self.Tension_N = {
            1: 73.0, 2: 70.0, 3: 68.0,
            4: 75.0, 5: 78.0, 6: 76.0
        }
        
        # フレット位置 (ナットからの距離)
        self.fret_pos = [
            self.scaleLength - (self.scaleLength / (2 ** (f / 12.0)))
            for f in range(self.f_max + 1)
        ]

    def calculate_physics_cost(self, string, fret):
        """
        Grimes (2014) Eq.7 に基づく押弦力(Fretting Force)の計算
        F = (T + EA * strain) * sin(theta)
        """
        # Action (a): 弦高 [mm]
        # 0フレットで0.5mm, 12フレットで2.5mm の線形モデル
        action_mm = 0.5 + (2.0 * fret / 12.0)
        
        # Vibrating Length (l): 振動弦長 [mm]
        l_vib_mm = self.scaleLength - self.fret_pos[fret]
        
        if l_vib_mm <= 0.001: return 0.0
        
        # --- SI単位系 (メートル) への変換 ---
        l_m = l_vib_mm / 1000.0
        a_m = action_mm / 1000.0
        
        # 物理定数の取得
        E = self.E_GPa[string] * 1.0e9  # GPa -> Pa (N/m^2)
        A = self.Area_m2[string]        # m^2
        T = self.Tension_N[string]      # N

        # 幾何計算 (直角三角形)
        # hyp = sqrt(l^2 + a^2)
        hyp_m = math.sqrt(l_m**2 + a_m**2)
        
        sin_theta = a_m / hyp_m
        cos_theta = l_m / hyp_m
        
        # 歪み項: (1 - cos) / cos = (1/cos) - 1
        strain_term = (1.0 / cos_theta) - 1.0
        
        # Grimes Eq.7: 押弦力 [N]
        # F_newton = (T + (EA * strain)) * sin(theta)
        F_newton = (T + (E * A * strain_term)) * sin_theta
        
        # --- コスト変換 ---
        # スケーリング係数 (OPLモデルに合わせた調整値: 15.0)
        # 以前のモデルと同程度のコスト感(最大10-15程度)にするための係数
        scalingFactor = 15.0
        physicsCost = F_newton * scalingFactor
        
        return physicsCost

# =============================================================================
# 2. 可視化実行
# =============================================================================
def plot_physics_cost_distribution():
    analyzer = PhysicsCostAnalyzer()
    
    # フレット範囲 (0〜22)
    frets = np.arange(0, 23, 1)
    
    # プロット設定
    plt.figure(figsize=(12, 7))
    
    # 各弦についてプロット (1弦〜6弦)
    colors = ['#FFD700', '#FFA500', '#FF4500', '#800080', '#0000FF', '#000080']
    labels = ['1st (High E)', '2nd (B)', '3rd (G)', '4th (D)', '5th (A)', '6th (Low E)']
    
    print("--- Physics Cost Calculation (Grimes 2014 Model) ---")
    print(f"Scaling Factor: 15.0")
    print(f"{'Fret':<5} | {'1st Str':<10} | {'6th Str':<10}")
    print("-" * 35)

    for s in range(1, 7):
        costs = []
        for f in frets:
            c = analyzer.calculate_physics_cost(s, f)
            costs.append(c)
            
        # グラフ描画
        lw = 1.0 + (s * 0.5)
        # ラベルに張力情報を付加
        label_text = f'{labels[s-1]} (T={analyzer.Tension_N[s]}N)'
        plt.plot(frets, costs, marker='o', markersize=4, linewidth=lw, 
                 label=label_text, color=colors[s-1])
        
        # データ確認用出力 (0, 12, 22フレットのみ)
        if s in [1, 6]:
            c0 = analyzer.calculate_physics_cost(1, 0)
            c12 = analyzer.calculate_physics_cost(1, 12)
            c22 = analyzer.calculate_physics_cost(1, 22)
            
            c0_6 = analyzer.calculate_physics_cost(6, 0)
            c12_6 = analyzer.calculate_physics_cost(6, 12)
            c22_6 = analyzer.calculate_physics_cost(6, 22)
            
    # 代表値の出力 (ループ外で一度だけ出力するため)
    c1_0 = analyzer.calculate_physics_cost(1, 0)
    c1_12 = analyzer.calculate_physics_cost(1, 12)
    c1_22 = analyzer.calculate_physics_cost(1, 22)
    c6_0 = analyzer.calculate_physics_cost(6, 0)
    c6_12 = analyzer.calculate_physics_cost(6, 12)
    c6_22 = analyzer.calculate_physics_cost(6, 22)
    
    print(f"{0:<5} | {c1_0:.4f}     | {c6_0:.4f}")
    print(f"{12:<5} | {c1_12:.4f}     | {c6_12:.4f}")
    print(f"{22:<5} | {c1_22:.4f}     | {c6_22:.4f}")

    plt.xlabel("Fret Number", fontsize=12)
    plt.ylabel("Cost Value", fontsize=12)
    
    plt.grid(True, which='both', linestyle='--', alpha=0.7)
    plt.legend(fontsize=10, loc='upper left')
    plt.xticks(np.arange(0, 23, 1))
    
    # ゾーンの注釈
    plt.axvspan(0, 5, color='green', alpha=0.05, label='Low Pos')
    plt.axvspan(12, 22, color='red', alpha=0.05, label='High Pos')
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_physics_cost_distribution()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Mock parameters for visualization (Normalized)
X_P = 5.0 # Pivot Fret Position (e.g., Fret 5)
Y_P = 0.0 # Pivot String Position (String 1, Y=0)
X_T = 10.0 # Target Fret Position (e.g., Fret 10)
Y_T = 3.0 # Target String Position (String 4, Y=3)
string_spacing = 1.0 # Normalized string spacing for axis labels

# Normalized Knuckle Distances (d_h) from Pivot P
# h=1 (Index): 0, h=2 (Middle): 1, h=3 (Ring): 2, h=4 (Pinky): 3
knuckle_dists = {1: 0.0, 2: 1.0, 3: 2.0, 4: 3.0}
finger_names = {1: 'Index (P)', 2: 'Middle', 3: 'Ring', 4: 'Pinky'}

# Example Wrist Angle
theta_deg = -30.0
theta_rad = np.deg2rad(theta_deg)

# Coordinates Calculation
P = (X_P, Y_P)
K_coords = {}
for h, d in knuckle_dists.items():
    K_x = X_P + d * np.cos(theta_rad)
    K_y = Y_P + d * np.sin(theta_rad)
    K_coords[h] = (K_x, K_y)
    
T = (X_T, Y_T)

# Knuckle Line (Collinearity)
knuckle_line_x = [K_coords[h][0] for h in range(1, 5)]
knuckle_line_y = [K_coords[h][1] for h in range(1, 5)]

# Plotting
fig, ax = plt.subplots(figsize=(8, 6))

# Plot String Lines (Y-axis grid)
max_string = 6
for s in range(1, max_string + 1):
    y = (s - 1) * string_spacing
    ax.axhline(y, color='gray', linestyle='--', linewidth=0.5)

# 1. Plot Knuckle Line (Collinearity)
ax.plot(knuckle_line_x, knuckle_line_y, ':', color='black', linewidth=1.0, label=r'Knuckle Line (Angle $\theta$)')

# 2. Plot Knuckles K_h
for h, (x, y) in K_coords.items():
    color = 'red' if h == 1 else 'blue'
    marker = 'o'
    ax.plot(x, y, marker, color=color, zorder=5)
    
    # Label Knuckles
    label_text = f'$K_{h}$'
    if h == 1:
        label_text += ' (P)'
    ax.text(x + 0.1, y + 0.1, label_text, color=color, fontsize=10)
        
# 3. Plot Target T
ax.plot(T[0], T[1], 's', color='green', zorder=5)
ax.text(T[0] + 0.1, T[1] + 0.1, r'$T$', color='green', fontsize=10)

# 4. Plot Stretch (D) - Example for Ring Finger (h=3)
K_target = K_coords[3]
V_x = T[0] - K_target[0]
V_y = T[1] - K_target[1]
D = np.sqrt(V_x**2 + V_y**2)

ax.plot([K_target[0], T[0]], [K_target[1], T[1]], '-', color='purple', linewidth=2, label=r'$D$ (Stretch Distance)')
ax.text((K_target[0] + T[0]) / 2 + 0.05, (K_target[1] + T[1]) / 2 + 0.15, r'$D$', color='purple', fontweight='bold')

# 5. Plot Wrist Angle (theta) arc
radius = 1.0
ax.arrow(P[0], P[1], radius, 0, head_width=0, head_length=0, fc='gray', ec='gray', linestyle='--')
arc = plt.matplotlib.patches.Arc(P, radius * 2, radius * 2, angle=0, theta1=theta_deg, theta2=0, color='gray', linestyle='-')
ax.add_patch(arc)
ax.text(P[0] + radius * np.cos(np.deg2rad(theta_deg / 2)) + 0.1, P[1] + radius * np.sin(np.deg2rad(theta_deg / 2)) - 0.2, r'$\theta$', color='black')

# Annotate axes and title
ax.set_xlabel('X軸: フレット数 (Fret Number)')
ax.set_ylabel('Y軸: 弦番号 (String Number)')
ax.set_title(r'Geometric Relationship for Stretch Cost ($ST$) and Knuckle Collinearity')

# Set Y-axis ticks to represent String Numbers (1-6)
y_ticks = [(s - 1) * string_spacing for s in range(1, max_string + 1)]
y_labels = [str(s) for s in range(1, max_string + 1)]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)

# Set limits and aspect ratio
ax.set_xlim(4.0, 11.0)
ax.set_ylim(-2.0, 5.5)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, linestyle='--')

# Save the figure
plt.savefig('stretch_cost_geometry_collinear.png')
print("stretch_cost_geometry_collinear.png")